In [1]:
# -*- coding: utf-8 -*-
"""
════════════════════════════════════════════════════════════════════════════
П.2 «Итогов второй встречи с оппонентом»
Вейвлет-разложение ряда RR на диапазоны НЧ (0,04–0,15 Гц) и ВЧ (0,15–0,4 Гц)

ЕДИНЫЙ ФАЙЛ. Ничего импортировать из соседних файлов не нужно.

Установка:
    pip install numpy scipy pyedflib PyWavelets matplotlib

Запуск из командной строки:
    python П2_ДВП_единый_код.py /путь/к/записям  [--out папка_для_результатов]

Запуск в Jupyter / Google Colab (выполнить ячейку целиком, затем):
    res = run_p2('/путь/к/записям', out_dir='results_p2')

В папке с записями должны лежать файлы вида 0011_ecg.edf ... 0030_ecg.edf
с исходными метками этапов пробы.

Что делает расчёт
  1. Детектирует R-зубцы в отведении II (детектор, общий для всех пунктов)
     и строит ряд RR.
  2. Окна фаз — по протоколу пробы (пассивный ортостаз на поворотном столе:
     около 5 мин лёжа, подъём стола до 45° без прерывания записи, около 5 мин
     в положении подъёма; метки «Этап» — начало подъёма около 05:00, конец
     подъёма около 05:10–05:20, конец записи около 10:00): покой — с 30 с до
     (начало подъёма − 5 с), наклон — с (конец подъёма + 15 с) по
     (конец подъёма + 275 с). Сам подъём и первые 15 с после него
     (переходный процесс) не используются.
     Первые 15 с после поворота стола (переходный процесс) не используются.
  3. Исправляет артефакты ряда RR: интервал, отличающийся от медианы
     11 соседних больше чем на 250 мс, удаляется и восполняется
     интерполяцией (порог «medium» по образцу Kubios HRV).
  4. Передискретизирует ряд кубическим сплайном на 4,8 Гц и удаляет
     линейный тренд.
  5. Раскладывает ряд вейвлет-пакетным преобразованием (дискретное
     вейвлет-преобразование с разложением и детализирующих, и
     аппроксимирующих коэффициентов), вейвлет dmey, уровень 6.
     При 4,8 Гц ширина одного узла 6-го уровня равна 0,0375 Гц, поэтому
     НЧ = узлы 1–3 = 0,0375–0,15 Гц, ВЧ = узлы 4–10 = 0,15–0,4125 Гц,
     то есть границы практически совпадают с заданными.
  6. Сравнивает результат с эталоном — интегрированием периодограммы
     Уэлча ровно в границах 0,04–0,15 и 0,15–0,4 Гц.
  7. Сравнивает пять опробованных вариантов разложения (обычное и пакетное
     ДВП, частоты 4 и 4,8 Гц, вейвлеты db4, sym8, dmey) по частотной
     характеристике и по согласию с эталоном; выбран лучший.
  8. Проверяет устойчивость выводов: без исправления артефактов
     и с окнами фаз целиком, включая переходные участки.
════════════════════════════════════════════════════════════════════════════
"""
!pip install numpy scipy pyedflib PyWavelets matplotlib
import argparse
import csv
import glob
import json
import os
import warnings

import numpy as np
import pyedflib
import pywt
from scipy import stats
from scipy.interpolate import CubicSpline
from scipy.ndimage import median_filter
from scipy.signal import butter, filtfilt, find_peaks, welch, detrend
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=UserWarning, module='pywt')

_trapz = getattr(np, 'trapezoid', None) or np.trapz     # NumPy 1.x и 2.x

# ─────────────────────────────────────────────────────────── параметры
CH_TILT = 1               # отведение II в записях ортостатической пробы
FS_RESAMPLE = 4.8         # Гц
LF_BAND = (0.04, 0.15)    # Гц, задано оппонентом
HF_BAND = (0.15, 0.40)    # Гц, задано оппонентом
WPD_WAVELET = 'dmey'
WPD_LEVEL = 6
ARTIFACT_MS = 250.0       # порог артефакта, мс
ARTIFACT_WIN = 11         # окно локальной медианы, интервалов
WELCH_SEG_S = 64.0        # длина сегмента Уэлча, с


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 1. Чтение записи, границы фаз, детектор R
# (детектор R общий для всех пунктов; границы фаз — по протоколу пробы)
# ═══════════════════════════════════════════════════════════════════════

def read_with_marks(path):
    f = pyedflib.EdfReader(path)
    fs = float(f.getSampleFrequency(CH_TILT))
    sig = f.readSignal(CH_TILT).astype(float)/1000.0     # мкВ -> мВ
    ann = f.readAnnotations()
    f.close()
    return sig, fs, [float(t) for t in ann[0]]


def phase_bounds(marks):
    """Границы подъёма стола по протоколу пробы. Метки «Этап»: около 05:00 —
    начало подъёма, около 05:10–05:20 — конец подъёма, около 10:00 — конец
    записи. Если в зоне подъёма несколько меток, первая — начало, последняя —
    конец; если одна и она позже 05:10 — это конец подъёма, начало принимается
    на 15 с раньше. Возвращает (начало подъёма, конец подъёма), с."""
    mid = sorted(t for t in marks if 250 < t < 340)
    if len(mid) >= 2:
        return mid[0], mid[-1]
    if len(mid) == 1:
        m = mid[0]
        return (m - 15.0, m) if m >= 310 else (m, m + 15.0)
    return 300.0, 315.0


def raise_marks_count(marks, merge_s=2.0):
    """Число различных меток в зоне подъёма (метки ближе merge_s секунд считаются
    одной). По протоколу их две: начало и конец подъёма."""
    mid = sorted(t for t in marks if 250 < t < 340)
    n = 0; last = None
    for t in mid:
        if last is None or t - last > merge_s:
            n += 1
        last = t
    return n


def detect_r(sig, fs, band=(5.0, 20.0), integ_ms=120,
             min_rr_ms=300, win_s=10.0, k=0.35, refine_ms=100):
    x = np.asarray(sig, float)
    x = x - np.median(x)

    # 1) полосовой фильтр — полоса энергии QRS
    ny = fs/2
    b, a = butter(3, [band[0]/ny, min(band[1]/ny, 0.99)], btype='band')
    xf = filtfilt(b, a, x)

    # 2) производная + возведение в квадрат (подчёркивает крутые фронты QRS)
    d = np.diff(xf, append=xf[-1])
    sq = d**2

    # 3) интегрирование скользящим окном ~ длительность QRS
    w = max(3, int(round(integ_ms/1000*fs)))
    integ = np.convolve(sq, np.ones(w)/w, mode='same')

    # 4) ЛОКАЛЬНО адаптивный порог: доля от локального максимума
    #    (окно win_s секунд, шаг — половина окна)
    n = len(integ)
    W = max(int(win_s*fs), w*4)
    thr = np.zeros(n)
    for start in range(0, n, W//2):
        end = min(start+W, n)
        seg = integ[start:end]
        if len(seg) == 0:
            continue
        local = k*np.percentile(seg, 98)      # 98-й процентиль ≈ уровень QRS
        thr[start:end] = np.maximum(thr[start:end], local)
    thr[thr == 0] = k*np.percentile(integ, 98)

    # 5) поиск пиков с рефрактерным периодом
    dist = max(1, int(round(min_rr_ms/1000*fs)))
    cand, _ = find_peaks(integ, height=thr, distance=dist)

    # 6) уточнение положения по максимуму |полосового сигнала|
    half = max(2, int(round(refine_ms/1000*fs/2)))
    out = []
    for c in cand:
        a0, b0 = max(0, c-half), min(len(xf), c+half+1)
        out.append(a0 + int(np.argmax(np.abs(xf[a0:b0]))))
    return np.array(sorted(set(out)), dtype=int), xf


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 2. Ряд RR, окна фаз, артефакты, передискретизация
# ═══════════════════════════════════════════════════════════════════════

def rr_series(path):
    """Ряд RR (мс) с моментами времени (с) и границами фаз одной записи."""
    sig, fs, marks = read_with_marks(path)
    p1e, p2b = phase_bounds(marks)
    n_marks = raise_marks_count(marks)
    r, _ = detect_r(sig, fs)
    t_r = r / fs
    rr = np.diff(r) / fs * 1000
    return dict(t_rr=t_r[:-1], rr=rr, p1e=p1e, p2b=p2b, n_marks=n_marks)


def phase_windows(v, mode='protocol'):
    """Окна фаз. 'protocol' — по протоколу пробы; 'full' — фазы целиком, с переходными участками."""
    if mode == 'protocol':
        return {'rest': (30.0, v['p1e'] - 5), 'tilt': (v['p2b'] + 15, v['p2b'] + 275)}
    return {'rest': (v['t_rr'][0], v['p1e']), 'tilt': (v['p2b'], v['t_rr'][-1])}


def artifact_mask(rr):
    """True для интервалов, отличающихся от локальной медианы > ARTIFACT_MS."""
    med = median_filter(rr, ARTIFACT_WIN, mode='nearest')
    return np.abs(rr - med) > ARTIFACT_MS


def uniform_segment(v, lo, hi, fs=FS_RESAMPLE, fix_artifacts=True):
    """Участок ряда RR внутри окна на равномерной сетке, без линейного тренда.
    Возвращает (x, число исправленных интервалов, число интервалов)."""
    t, rr = v['t_rr'], v['rr']
    inside = (t >= lo) & (t <= hi)
    bad = artifact_mask(rr) if fix_artifacts else np.zeros_like(inside)
    keep = inside & ~bad
    cs = CubicSpline(t[keep], rr[keep])
    tu = np.arange(t[inside][0], t[inside][-1], 1 / fs)
    return detrend(cs(tu), type='linear'), int((inside & bad).sum()), int(inside.sum())


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 3. Методы оценки мощности в полосах
# ═══════════════════════════════════════════════════════════════════════

def welch_power(x, fs, band):
    """Эталон: интеграл периодограммы Уэлча ровно в границах полосы."""
    f, p = welch(x, fs=fs, nperseg=min(int(WELCH_SEG_S * fs), len(x)))
    m = (f >= band[0]) & (f < band[1])
    return float(_trapz(p[m], f[m]))


def wpd_power(x, fs=FS_RESAMPLE, wavelet=WPD_WAVELET, level=WPD_LEVEL,
              lf=LF_BAND, hf=HF_BAND):
    """Вейвлет-пакетное разложение. Узлы уровня level упорядочены по частоте;
    полоса набирается из узлов, центр которых попадает в заданные границы.
    Мощность полосы = дисперсия сигнала, восстановленного только из её узлов.
    Возвращает (мощность НЧ, мощность ВЧ, фактические границы НЧ, ВЧ)."""
    wp = pywt.WaveletPacket(x, wavelet, mode='symmetric', maxlevel=level)
    nodes = wp.get_level(level, 'freq')
    bw = fs / 2 / 2 ** level

    def pick(band):
        return [k for k in range(len(nodes)) if band[0] <= (k + 0.5) * bw < band[1]]

    def power(ks):
        nw = pywt.WaveletPacket(None, wavelet, mode='symmetric', maxlevel=level)
        for k in ks:
            nw[nodes[k].path] = nodes[k].data
        return float(nw.reconstruct(update=False)[:len(x)].var())

    k_lf, k_hf = pick(lf), pick(hf)
    edges = lambda ks: (ks[0] * bw, (ks[-1] + 1) * bw)
    return power(k_lf), power(k_hf), edges(k_lf), edges(k_hf)


def dwt_old_power(x, fs=4.0, wavelet='db4', level=6):
    """Для сравнения: обычное ДВП при 4 Гц, НЧ = cD5+cD6 (0,031–0,125 Гц),
    ВЧ = cD3+cD4 (0,125–0,5 Гц). pywt.wavedec отдаёт [cA6, cD6, cD5, ..., cD1]."""
    c = pywt.wavedec(x, wavelet, level=level, mode='symmetric')

    def power(levels):
        keep = {level - j + 1 for j in levels}
        z = [cc if i in keep else np.zeros_like(cc) for i, cc in enumerate(c)]
        return float(pywt.waverec(z, wavelet, mode='symmetric')[:len(x)].var())

    return power([5, 6]), power([3, 4])


def dwt_power(x, fs, wavelet, level, lf_levels, hf_levels):
    """Обычное ДВП: мощность НЧ и ВЧ как дисперсия сигнала, восстановленного
    из заданных уровней детализации (номера уровней — истинные, cD1 — самый
    высокочастотный)."""
    c = pywt.wavedec(x, wavelet, level=level, mode='symmetric')

    def power(levels):
        keep = {level - j + 1 for j in levels}
        z = [cc if i in keep else np.zeros_like(cc) for i, cc in enumerate(c)]
        return float(pywt.waverec(z, wavelet, mode='symmetric')[:len(x)].var())

    return power(lf_levels), power(hf_levels)


# Опробованные варианты разложения: (название, частота, функция, полосы НЧ, ВЧ)
VARIANTS = [
    ('Обычное ДВП, 4 Гц, db4', 4.0,
     lambda x: dwt_power(x, 4.0, 'db4', 6, [5, 6], [3, 4]), (0.03125, 0.125), (0.125, 0.5)),
    ('Обычное ДВП, 4,8 Гц, db4', 4.8,
     lambda x: dwt_power(x, 4.8, 'db4', 6, [5, 6], [3, 4]), (0.0375, 0.15), (0.15, 0.6)),
    ('Пакетное ДВП, 4,8 Гц, db4', 4.8,
     lambda x: wpd_power(x, wavelet='db4')[:2], (0.0375, 0.15), (0.15, 0.4125)),
    ('Пакетное ДВП, 4,8 Гц, sym8', 4.8,
     lambda x: wpd_power(x, wavelet='sym8')[:2], (0.0375, 0.15), (0.15, 0.4125)),
    ('Пакетное ДВП, 4,8 Гц, dmey', 4.8,
     lambda x: wpd_power(x, wavelet='dmey')[:2], (0.0375, 0.15), (0.15, 0.4125)),
]


def variant_response(func, fs, duration=260.0, n_phase=10, seed=0):
    """Показатели частотной характеристики варианта по синусоидам со
    случайной фазой: минимальная доля, отнесённая к своему диапазону
    в его внутренней части (НЧ 0,06–0,13 Гц, ВЧ 0,17–0,38 Гц), и
    максимальная доля, попадающая в анализ с частот выше 0,43 Гц."""
    rng = np.random.default_rng(seed)
    t = np.arange(0, duration, 1 / fs)

    def share(f0):
        lf = hf = 0.0
        for _ in range(n_phase):
            x = np.sin(2 * np.pi * f0 * t + rng.uniform(0, 2 * np.pi))
            a, b = func(x)
            lf += a / x.var() / n_phase; hf += b / x.var() / n_phase
        return lf, hf

    lf_core = min(share(f)[0] for f in np.arange(0.06, 0.131, 0.01))
    hf_core = min(share(f)[1] for f in np.arange(0.17, 0.381, 0.03))
    outside = max(sum(share(f)) for f in (0.45, 0.5, 0.55))
    return dict(lf_core=float(lf_core), hf_core=float(hf_core), outside=float(outside))


def compare_variants(series):
    """Сравнение всех вариантов: частотная характеристика и согласие
    отношения НЧ/ВЧ с эталоном (Уэлч) на реальных записях."""
    names = sorted(series)
    ref, est = {}, {i: {} for i in range(len(VARIANTS))}
    for n in names:
        v = series[n]
        for phase, (lo, hi) in phase_windows(v).items():
            x48, _, _ = uniform_segment(v, lo, hi, fs=4.8)
            x4, _, _ = uniform_segment(v, lo, hi, fs=4.0)
            ref[(n, phase)] = (welch_power(x48, 4.8, LF_BAND), welch_power(x48, 4.8, HF_BAND))
            for i, (_, fs, func, _, _) in enumerate(VARIANTS):
                est[i][(n, phase)] = func(x48 if fs == 4.8 else x4)
    out = []
    for i, (title, fs, func, lfb, hfb) in enumerate(VARIANTS):
        row = dict(title=title, lf_band=lfb, hf_band=hfb, **variant_response(func, fs))
        for phase in ('rest', 'tilt'):
            r = [ref[(n, phase)][0] / ref[(n, phase)][1] for n in names]
            e = [est[i][(n, phase)][0] / est[i][(n, phase)][1] for n in names]
            row[f'rho_{phase}'] = float(stats.spearmanr(r, e)[0])
            row[f'bias_{phase}'] = float(np.median(np.array(e) / np.array(r)))
        for k, q in enumerate(('lf', 'hf')):
            d_ref = [np.sign(ref[(n, 'tilt')][k] - ref[(n, 'rest')][k]) for n in names]
            d_est = [np.sign(est[i][(n, 'tilt')][k] - est[i][(n, 'rest')][k]) for n in names]
            row[f'direction_{q}'] = int(sum(a == b for a, b in zip(d_ref, d_est)))
        out.append(row)
    return out


def frequency_response(freqs=None, duration=260.0):
    """Доля мощности синусоиды, попадающая в НЧ и ВЧ, для пакетного
    и обычного ДВП. Это прямая проверка того, какие частоты метод относит к полосам."""
    if freqs is None:
        freqs = np.round(np.arange(0.01, 0.601, 0.005), 3)
    out = {'freq': freqs.tolist(), 'new_lf': [], 'new_hf': [], 'old_lf': [], 'old_hf': []}
    for f0 in freqs:
        t = np.arange(0, duration, 1 / FS_RESAMPLE)
        x = np.sin(2 * np.pi * f0 * t)
        lf, hf, _, _ = wpd_power(x)
        out['new_lf'].append(lf / x.var()); out['new_hf'].append(hf / x.var())
        t4 = np.arange(0, duration, 1 / 4.0)
        x4 = np.sin(2 * np.pi * f0 * t4)
        lf4, hf4 = dwt_old_power(x4)
        out['old_lf'].append(lf4 / x4.var()); out['old_hf'].append(hf4 / x4.var())
    return out


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 4. Расчёт по всем записям
# ═══════════════════════════════════════════════════════════════════════

def analyse(series, windows='protocol', fix_artifacts=True):
    """Мощности НЧ/ВЧ всеми методами для каждой записи и фазы."""
    rows = []
    for name, v in sorted(series.items()):
        rec = {'record': name}
        for phase, (lo, hi) in phase_windows(v, windows).items():
            x, n_fix, n_rr = uniform_segment(v, lo, hi, fix_artifacts=fix_artifacts)
            lf_w, hf_w = welch_power(x, FS_RESAMPLE, LF_BAND), welch_power(x, FS_RESAMPLE, HF_BAND)
            lf_p, hf_p, e_lf, e_hf = wpd_power(x)
            x4, _, _ = uniform_segment(v, lo, hi, fs=4.0, fix_artifacts=fix_artifacts)
            lf_o, hf_o = dwt_old_power(x4)
            rec.update({f'{phase}_lf_welch': lf_w, f'{phase}_hf_welch': hf_w,
                        f'{phase}_lf_wpd': lf_p, f'{phase}_hf_wpd': hf_p,
                        f'{phase}_lf_old': lf_o, f'{phase}_hf_old': hf_o,
                        f'{phase}_fixed': n_fix, f'{phase}_n_rr': n_rr,
                        f'{phase}_len_s': len(x) / FS_RESAMPLE})
        rec['wpd_edges'] = (e_lf, e_hf)
        rows.append(rec)
    return rows


def holm(pvals):
    """Поправка Холма на множественные сравнения."""
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj


def summarise(rows):
    """Изменение при наклоне и согласие методов."""
    get = lambda key: np.array([r[key] for r in rows])
    s = {'n': len(rows), 'tests': [], 'agreement': {}}
    for method in ('welch', 'wpd', 'old'):
        for q in ('lf', 'hf', 'lfhf'):
            if q == 'lfhf':
                rest = get(f'rest_lf_{method}') / get(f'rest_hf_{method}')
                tilt = get(f'tilt_lf_{method}') / get(f'tilt_hf_{method}')
            else:
                rest, tilt = get(f'rest_{q}_{method}'), get(f'tilt_{q}_{method}')
            lr = np.log(tilt / rest)
            s['tests'].append(dict(method=method, quantity=q,
                                   median_rest=float(np.median(rest)), median_tilt=float(np.median(tilt)),
                                   median_ratio=float(np.exp(np.median(lr))),
                                   n_up=int((lr > 0).sum()), p=float(stats.wilcoxon(lr)[1])))
    main = [t for t in s['tests'] if t['method'] in ('welch', 'wpd')]
    for t, pa in zip(main, holm([t['p'] for t in main])):
        t['p_holm'] = float(pa)
    for method in ('wpd', 'old'):
        a = {}
        for phase in ('rest', 'tilt'):
            ref = get(f'{phase}_lf_welch') / get(f'{phase}_hf_welch')
            est = get(f'{phase}_lf_{method}') / get(f'{phase}_hf_{method}')
            rho, p = stats.spearmanr(ref, est)
            a[phase] = dict(rho=float(rho), p=float(p),
                            ratio_to_ref=float(np.median(est / ref)))
        for q in ('lf', 'hf'):
            d_ref = np.sign(get(f'tilt_{q}_welch') - get(f'rest_{q}_welch'))
            d_est = np.sign(get(f'tilt_{q}_{method}') - get(f'rest_{q}_{method}'))
            a[f'direction_{q}'] = int((d_ref == d_est).sum())
        s['agreement'][method] = a
    return s


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 5. Рисунки
# ═══════════════════════════════════════════════════════════════════════

C_LF, C_HF, C_REST, C_TILT = '#2e7d32', '#e65100', '#1565c0', '#c62828'


def _comma_axes(fig):
    """Десятичная запятая на линейных осях (русская запись чисел)."""
    from matplotlib.ticker import FuncFormatter
    fmt = FuncFormatter(lambda v, _: f'{v:g}'.replace('.', ','))
    for a in fig.axes:
        if a.get_xscale() == 'linear':
            a.xaxis.set_major_formatter(fmt)
        if a.get_yscale() == 'linear':
            a.yaxis.set_major_formatter(fmt)


def fig_response(resp, path):
    f = np.array(resp['freq'])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for a, key, title in [(ax[0], 'old', 'а — обычное ДВП, 4 Гц, db4'),
                          (ax[1], 'new', 'б — пакетное ДВП, 4,8 Гц, dmey')]:
        a.axvspan(*LF_BAND, color=C_LF, alpha=0.10)
        a.axvspan(*HF_BAND, color=C_HF, alpha=0.10)
        a.plot(f, resp[f'{key}_lf'], color=C_LF, lw=2, label='доля, отнесённая к НЧ')
        a.plot(f, resp[f'{key}_hf'], color=C_HF, lw=2, label='доля, отнесённая к ВЧ')
        a.set_title(title, fontsize=11); a.set_xlabel('частота синусоиды, Гц')
        a.set_xlim(0, 0.6); a.set_ylim(-0.03, 1.12); a.grid(alpha=0.3)
    ax[0].set_ylabel('доля мощности'); ax[0].legend(loc='center right', fontsize=9)
    fig.text(0.5, 0.005, 'Заливка — заданные полосы: НЧ 0,04–0,15 Гц (зелёная), ВЧ 0,15–0,4 Гц (оранжевая)',
             ha='center', fontsize=9)
    _comma_axes(fig); fig.tight_layout(rect=(0, 0.04, 1, 1)); fig.savefig(path, dpi=150); plt.close(fig)


def fig_spectra(series, names, path):
    fig, ax = plt.subplots(1, len(names), figsize=(12, 4.2))
    for a, name in zip(np.atleast_1d(ax), names):
        v = series[name]
        lo_all = []
        for phase, col, lab in [('rest', C_REST, 'покой'), ('tilt', C_TILT, 'наклон')]:
            lo, hi = phase_windows(v)[phase]
            x, _, _ = uniform_segment(v, lo, hi)
            f, p = welch(x, fs=FS_RESAMPLE, nperseg=int(WELCH_SEG_S * FS_RESAMPLE))
            m = f <= 0.5
            a.semilogy(f[m], p[m], color=col, lw=1.8, label=lab); lo_all.append(p[m][f[m] > 0.02].min())
        a.axvspan(*LF_BAND, color=C_LF, alpha=0.10); a.axvspan(*HF_BAND, color=C_HF, alpha=0.10)
        a.set_ylim(min(lo_all) / 3, None); a.set_xlim(0, 0.5)
        a.set_title(f'запись {name}', fontsize=11); a.set_xlabel('частота, Гц')
        a.set_ylabel('спектральная плотность, мс²/Гц'); a.grid(alpha=0.3, which='both'); a.legend(fontsize=9)
    _comma_axes(fig); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def fig_agreement(rows, s, path):
    get = lambda key: np.array([r[key] for r in rows])
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    for a, method, title in [(ax[0], 'old', 'а — обычное ДВП, 4 Гц'),
                             (ax[1], 'wpd', 'б — пакетное ДВП, 4,8 Гц')]:
        allv = []
        for phase, col, lab in [('rest', C_REST, 'покой'), ('tilt', C_TILT, 'наклон')]:
            ref = get(f'{phase}_lf_welch') / get(f'{phase}_hf_welch')
            est = get(f'{phase}_lf_{method}') / get(f'{phase}_hf_{method}')
            rho = s['agreement'][method][phase]['rho']
            a.scatter(ref, est, color=col, s=36, alpha=0.85, label=f'{lab}: ρ = {rho:.2f}'.replace('.', ','))
            allv += [ref, est]
        allv = np.concatenate(allv); lim = (allv.min() / 1.4, allv.max() * 1.4)
        a.plot(lim, lim, 'k--', lw=0.8); a.set_xscale('log'); a.set_yscale('log')
        a.set_xlim(lim); a.set_ylim(lim); a.set_aspect('equal')
        a.set_xlabel('НЧ/ВЧ, эталон (Уэлч)'); a.set_ylabel('НЧ/ВЧ, вейвлет-оценка')
        a.set_title(title, fontsize=11); a.grid(alpha=0.3, which='both'); a.legend(fontsize=9)
    _comma_axes(fig); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def fig_paired(rows, path):
    get = lambda key: np.array([r[key] for r in rows])
    names = [r['record'] for r in rows]
    fig, ax = plt.subplots(figsize=(12, 4.6))
    x = np.arange(len(rows))
    for method, off, mk, lab in [('welch', -0.12, 'o', 'эталон (Уэлч)'), ('wpd', 0.12, 's', 'пакетное ДВП')]:
        rest = get(f'rest_lf_{method}') / get(f'rest_hf_{method}')
        tilt = get(f'tilt_lf_{method}') / get(f'tilt_hf_{method}')
        for i in x:
            ax.plot([i + off, i + off], [rest[i], tilt[i]], color='0.6', lw=1)
        ax.scatter(x + off, rest, marker=mk, color=C_REST, s=30, zorder=3,
                   label=f'покой, {lab}')
        ax.scatter(x + off, tilt, marker=mk, color=C_TILT, s=30, zorder=3,
                   label=f'наклон, {lab}')
    ax.set_yscale('log'); ax.set_xticks(x); ax.set_xticklabels(names, rotation=90)
    ax.set_xlabel('запись'); ax.set_ylabel('НЧ/ВЧ'); ax.grid(alpha=0.3, which='both')
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)   # по X — номера записей


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 6. Общий запуск
# ═══════════════════════════════════════════════════════════════════════

NAMES = {'welch': 'Уэлч (эталон)', 'wpd': 'пакетное ДВП', 'old': 'обычное ДВП'}
QNAMES = {'lf': 'НЧ', 'hf': 'ВЧ', 'lfhf': 'НЧ/ВЧ'}


def _print_summary(s, title):
    print('\n' + '=' * 92); print(title); print('=' * 92)
    print(f"{'метод':16s}{'величина':10s}{'покой':>10s}{'наклон':>10s}{'наклон/покой':>14s}"
          f"{'рост, из ' + str(s['n']):>12s}{'p':>11s}{'p (Холм)':>11s}")
    for t in s['tests']:
        ph = f"{t['p_holm']:.2g}" if 'p_holm' in t else '—'
        print(f"{NAMES[t['method']]:16s}{QNAMES[t['quantity']]:10s}{t['median_rest']:10.3g}"
              f"{t['median_tilt']:10.3g}{t['median_ratio']:14.2f}{t['n_up']:12d}{t['p']:11.2g}{ph:>11s}")
    for method in ('wpd', 'old'):
        a = s['agreement'][method]
        print(f"Согласие с эталоном, {NAMES[method]}: ρ(покой) = {a['rest']['rho']:.2f}, "
              f"ρ(наклон) = {a['tilt']['rho']:.2f}; направление НЧ {a['direction_lf']}/{s['n']}, "
              f"ВЧ {a['direction_hf']}/{s['n']}")


def run_p2(data_dir, out_dir='results_p2'):
    os.makedirs(out_dir, exist_ok=True)
    paths = sorted(glob.glob(os.path.join(data_dir, '00*_ecg.edf')))
    if not paths:
        raise SystemExit(f'В папке {data_dir} не найдено файлов вида 00*_ecg.edf')
    print(f'Записей: {len(paths)}. Детекция R-зубцов...')
    series = {os.path.basename(p)[:4]: rr_series(p) for p in paths}

    for name, v in series.items():
        if v['n_marks'] != 2:
            print(f'  Запись {name}: меток в зоне подъёма стола — {v["n_marks"]} (по протоколу две); '
                  f'начало подъёма принято {v["p1e"]:.0f} с, конец подъёма {v["p2b"]:.0f} с')

    rows = analyse(series)
    s = summarise(rows)
    e_lf, e_hf = rows[0]['wpd_edges']
    print(f'\nПакетное ДВП ({WPD_WAVELET}, уровень {WPD_LEVEL}, {FS_RESAMPLE} Гц): '
          f'НЧ = {e_lf[0]:.4f}–{e_lf[1]:.4f} Гц, ВЧ = {e_hf[0]:.4f}–{e_hf[1]:.4f} Гц')
    fixed = [(r['record'], r['rest_fixed'], r['tilt_fixed']) for r in rows if r['rest_fixed'] + r['tilt_fixed']]
    print('Исправлено артефактных интервалов (покой, наклон): ' +
          ', '.join(f'{n} ({a}, {b})' for n, a, b in fixed))
    _print_summary(s, 'ОСНОВНОЙ РАСЧЁТ: окна по протоколу пробы, артефакты исправлены')

    sens = {'без исправления артефактов': summarise(analyse(series, fix_artifacts=False)),
            'фазы целиком, с переходными участками': summarise(analyse(series, windows='full'))}
    complete = {k: v for k, v in series.items() if v['n_marks'] == 2}
    if len(complete) < len(series):
        sens['без записей с нестандартной разметкой подъёма'] = summarise(analyse(complete))
    for title, ss in sens.items():
        _print_summary(ss, 'ПРОВЕРКА УСТОЙЧИВОСТИ: ' + title)

    variants = compare_variants(series)
    print('\n' + '=' * 92); print('СРАВНЕНИЕ ОПРОБОВАННЫХ ВАРИАНТОВ РАЗЛОЖЕНИЯ'); print('=' * 92)
    print(f"{'вариант':30s}{'НЧ, Гц':>16s}{'ВЧ, Гц':>16s}{'НЧ ядро':>9s}{'ВЧ ядро':>9s}{'>0,43':>7s}"
          f"{'ρ пок.':>8s}{'ρ накл.':>8s}{'напр НЧ':>8s}")
    for r in variants:
        print(f"{r['title']:30s}{r['lf_band'][0]:7.4f}–{r['lf_band'][1]:<8.4f}{r['hf_band'][0]:7.4f}–{r['hf_band'][1]:<8.4f}"
              f"{r['lf_core']:9.2f}{r['hf_core']:9.2f}{r['outside']:7.2f}{r['rho_rest']:8.2f}{r['rho_tilt']:8.2f}"
              f"{r['direction_lf']:8d}")

    resp = frequency_response()
    fig_response(resp, os.path.join(out_dir, 'p2_fig1_freq_response.png'))
    fig_spectra(series, ['0012', '0019'], os.path.join(out_dir, 'p2_fig2_spectra.png'))
    fig_agreement(rows, s, os.path.join(out_dir, 'p2_fig3_agreement.png'))
    fig_paired(rows, os.path.join(out_dir, 'p2_fig4_per_record.png'))

    keys = [k for k in rows[0] if k != 'wpd_edges']
    with open(os.path.join(out_dir, 'p2_per_record.csv'), 'w', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=keys, extrasaction='ignore'); w.writeheader(); w.writerows(rows)
    with open(os.path.join(out_dir, 'p2_summary.json'), 'w', encoding='utf-8') as fh:
        json.dump({'main': s, 'sensitivity': sens, 'variants': variants, 'wpd_edges': rows[0]['wpd_edges'],
                   'fixed': fixed, 'frequency_response': resp}, fh, ensure_ascii=False, indent=1)
    print(f'\nРезультаты сохранены в папку {out_dir}')
    return dict(rows=rows, summary=s, sensitivity=sens, variants=variants, response=resp)


def _in_notebook():
    """True, если код выполняется в Jupyter / Google Colab (там нельзя разбирать
    аргументы командной строки: ядро передаёт свои служебные параметры)."""
    try:
        get_ipython  # noqa: F821  — определена только внутри Jupyter / Colab
        return True
    except NameError:
        return False


if __name__ == '__main__' and not _in_notebook():
    ap = argparse.ArgumentParser(description='П.2: вейвлет-разложение ряда RR')
    ap.add_argument('data_dir', help='папка с записями 00*_ecg.edf')
    ap.add_argument('--out', default='results_p2', help='папка для результатов')
    a = ap.parse_args()
    run_p2(a.data_dir, a.out)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 11.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyedflib: filename=pyedflib-0.1.42-cp313-cp313-linux_x86_64.whl size=2852053 sha256=7312b259923a9ee7f3f950ed516881765aeed59d87a07ea4162a96d4214c1ded
  Stored in directory: /root/.cache/pip/wheels/ac/16/3e/5299c8a6b023bfd3d7204240ca5de940a49bede1edb30015ab
Successfully built pyedflib


In [2]:
run_p2('/content/data/')

Записей: 20. Детекция R-зубцов...
  Запись 0023: меток в зоне подъёма стола — 1 (по протоколу две); начало подъёма принято 306 с, конец подъёма 321 с
  Запись 0028: меток в зоне подъёма стола — 3 (по протоколу две); начало подъёма принято 303 с, конец подъёма 320 с

Пакетное ДВП (dmey, уровень 6, 4.8 Гц): НЧ = 0.0375–0.1500 Гц, ВЧ = 0.1500–0.4125 Гц
Исправлено артефактных интервалов (покой, наклон): 0011 (16, 0), 0022 (2, 0), 0025 (1, 7), 0026 (5, 5), 0028 (1, 0), 0029 (4, 0)

ОСНОВНОЙ РАСЧЁТ: окна по протоколу пробы, артефакты исправлены
метод           величина       покой    наклон  наклон/покой рост, из 20          p   p (Холм)
Уэлч (эталон)   НЧ               432       487          1.38          14      0.036      0.036
Уэлч (эталон)   ВЧ               927       282          0.46           2    1.9e-05    7.6e-05
Уэлч (эталон)   НЧ/ВЧ          0.491       1.4          3.18          19    3.8e-06    2.3e-05
пакетное ДВП    НЧ               493       664          1.36          15   

{'rows': [{'record': '0011',
   'rest_lf_welch': 1038.5522236957095,
   'rest_hf_welch': 6543.819920229089,
   'rest_lf_wpd': 1737.8657564163643,
   'rest_hf_wpd': 6708.6355282791565,
   'rest_lf_old': 1236.4921237549152,
   'rest_hf_old': 7231.646530355713,
   'rest_fixed': 16,
   'rest_n_rr': 273,
   'rest_len_s': 266.0416666666667,
   'tilt_lf_welch': 519.9491477748077,
   'tilt_hf_welch': 429.9264779840631,
   'tilt_lf_wpd': 710.6461764294015,
   'tilt_hf_wpd': 604.4339520440615,
   'tilt_lf_old': 492.67526528766723,
   'tilt_hf_old': 770.0550040390779,
   'tilt_fixed': 0,
   'tilt_n_rr': 355,
   'tilt_len_s': 259.375,
   'wpd_edges': ((0.0375, 0.15), (0.15, 0.4125))},
  {'record': '0012',
   'rest_lf_welch': 161.85705611160094,
   'rest_hf_welch': 777.9314772940622,
   'rest_lf_wpd': 185.16941583373062,
   'rest_hf_wpd': 843.9913391212672,
   'rest_lf_old': 177.96792878275943,
   'rest_hf_old': 821.0485931700916,
   'rest_fixed': 0,
   'rest_n_rr': 253,
   'rest_len_s': 265.208333